# Data Science Practice Notebook — Part 2 (Extensive Edition)
### Lecture 3: Pandas II  •  Lecture 4: Seaborn & EDA

**Level:** Beginner → Intermediate

**How to use this notebook**
- Read the concept notes, then **run** every worked example.
- Worked examples use Seaborn's **built-in datasets** — they run offline, no downloads.
- Watch for the ⚠️ **Common Mistakes** cells.
- Tasks have empty cells for you; full answers are in the separate *Solutions* notebook.

**Before the Tasks:** download the **Wine Quality** dataset from Kaggle
(https://www.kaggle.com/datasets/uciml/red-wine-quality-cortez-et-al-2009),
save the red-wine file as **`winequality-red.csv`** next to this notebook.
⚠️ It is **semicolon-separated** — load with `sep=';'`.


---
# Lecture 3 — Pandas II

The two power tools of real analysis: **`groupby`** (summarise by category) and **`merge`/`join`** (combine tables). Plus reshaping with `pivot`/`melt`.

In [7]:
import pandas as pd, numpy as np, seaborn as sns
tips = sns.load_dataset("tips")
tips.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


## 3.1 GroupBy — Split · Apply · Combine

In [ ]:
tips.groupby("day")["total_bill"].mean()

In [ ]:
tips.groupby("day")["total_bill"].agg(["mean", "max", "count"])

In [ ]:
# Multiple columns grouped
tips.groupby(["day", "sex"])["tip"].mean()

In [ ]:
# Named aggregations across different columns
tips.groupby("day").agg(
    avg_bill=("total_bill", "mean"),
    total_tips=("tip", "sum"),
    n=("size", "count"),
)

### `transform` vs `agg`
`agg` collapses each group to one row; `transform` returns a value for **every** row.

In [ ]:
tips = tips.copy()
tips["day_avg"] = tips.groupby("day")["total_bill"].transform("mean")
tips["pct_of_day_avg"] = (tips["total_bill"] / tips["day_avg"] * 100).round(1)
tips[["day", "total_bill", "day_avg", "pct_of_day_avg"]].head()

### `filter` — keep whole groups by a condition

In [ ]:
# Keep only days that have more than 60 records
big_days = tips.groupby("day").filter(lambda g: len(g) > 60)
print(big_days["day"].value_counts())

⚠️ **Common Mistake 8.** By default `groupby` drops `NaN` keys and returns the group columns as the **index**. Use `as_index=False` (or `.reset_index()`) if you want a flat DataFrame back.

In [ ]:
flat = tips.groupby("day", as_index=False, observed=True)["tip"].mean()
print(flat)

## 3.2 Combining Tables — merge / join / concat

In [ ]:
customers = pd.DataFrame({
    "cust_id": [1, 2, 3, 4],
    "name": ["Ali", "Sara", "Zed", "Mina"],
    "city": ["Lahore", "Karachi", "Lahore", "Quetta"],
})
orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105],
    "cust_id": [1, 2, 2, 3, 99],
    "amount": [250, 100, 400, 150, 300],
})
print(customers); print(orders)

In [ ]:
print("inner:\n", pd.merge(customers, orders, on="cust_id", how="inner"))

In [ ]:
print("left:\n", pd.merge(customers, orders, on="cust_id", how="left"))

In [ ]:
print("outer (indicator shows origin):\n",
      pd.merge(customers, orders, on="cust_id", how="outer", indicator=True))

### Merging on differently-named keys, and handling overlaps with suffixes

In [ ]:
left = pd.DataFrame({"id": [1, 2, 3], "val": [10, 20, 30]})
right = pd.DataFrame({"key": [1, 2, 4], "val": [100, 200, 400]})
merged = pd.merge(left, right, left_on="id", right_on="key",
                  how="inner", suffixes=("_left", "_right"))
print(merged)

### `concat` — stack rows or columns

In [ ]:
jan = pd.DataFrame({"item": ["A", "B"], "sales": [10, 20]})
feb = pd.DataFrame({"item": ["A", "B"], "sales": [15, 25]})
print(pd.concat([jan, feb], keys=["Jan", "Feb"]))

⚠️ **Common Mistake 9.** After a **left/outer** merge, unmatched rows contain `NaN`. Check with `.isnull().sum()` before assuming every row matched — silent `NaN`s break later calculations.

## 3.3 Reshaping — pivot_table & melt

In [ ]:
tips.pivot_table(values="total_bill", index="day",
                 columns="sex", aggfunc="mean", observed=True)

In [ ]:
# melt: wide -> long
wide = pd.DataFrame({"name": ["Ali", "Sara"], "math": [80, 90], "science": [75, 85]})
print(wide)
print("\nmelted:\n", wide.melt(id_vars="name", var_name="subject", value_name="marks"))

---
## 🧪 Lecture 3 Tasks — Pandas II (Wine Quality)

Load the data (semicolon-separated!) then solve. **Drills:** 1–7  •  **Challenges:** 8–11


In [18]:
import pandas as pd
import numpy as np
import os

# Exact path of your dataset file
file_path = r"C:\Users\HP\Downloads\archive\winequality-red.csv"

# Check if file exists in path, otherwise fallback to local folder
if os.path.exists(file_path):
    wine = pd.read_csv(file_path)
else:
    wine = pd.read_csv("winequality-red.csv")

# Verify that all 12 columns are loaded properly
print("Columns Loaded Successfully:", wine.columns.tolist())

Columns Loaded Successfully: ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality']


**Task 1 (drill).** Group by `quality`, compute mean `alcohol`. Which quality has the highest?

In [25]:
# Task 1 Code
avg_alc = wine.groupby("quality")["alcohol"].mean()
print(avg_alc)
print(f"\nHighest quality alcohol: {avg_alc.idxmax()} with mean {avg_alc.max():.2f}")

quality
3     9.955000
4    10.265094
5     9.899706
6    10.629519
7    11.465913
8    12.094444
Name: alcohol, dtype: float64

Highest quality alcohol: 8 with mean 12.09


**Task 2 (drill).** For each `quality`, report mean/min/max of `pH` with `.agg()`.

In [26]:
# Task 2 Code
ph_stats = wine.groupby("quality")["pH"].agg(["mean", "min", "max"])
print(ph_stats)

             mean   min   max
quality                      
3        3.398000  3.16  3.63
4        3.381509  2.74  3.90
5        3.304949  2.88  3.74
6        3.318072  2.86  4.01
7        3.290754  2.92  3.78
8        3.267222  2.88  3.72


**Task 3 (drill).** Add `quality_label` = 'high' if quality>=7 else 'low'; show its counts.

In [27]:
# Task 3 Code
wine["quality_label"] = np.where(wine["quality"] >= 7, "high", "low")
print(wine["quality_label"].value_counts())

quality_label
low     1382
high     217
Name: count, dtype: int64


**Task 4 (drill).** Group by `quality` and report the mean of `alcohol`, `sulphates`, and `pH` together.

In [28]:
# Task 4 Code
multi_means = wine.groupby("quality")[["alcohol", "sulphates", "pH"]].mean()
print(multi_means)

           alcohol  sulphates        pH
quality                                
3         9.955000   0.570000  3.398000
4        10.265094   0.596415  3.381509
5         9.899706   0.620969  3.304949
6        10.629519   0.675329  3.318072
7        11.465913   0.741256  3.290754
8        12.094444   0.767778  3.267222


**Task 5 (drill).** Use `as_index=False` to get a flat table of mean `residual sugar` per `quality`.

In [29]:
# Task 5 Code
flat_sugar = wine.groupby("quality", as_index=False)["residual sugar"].mean()
print(flat_sugar)

   quality  residual sugar
0        3        2.635000
1        4        2.694340
2        5        2.528855
3        6        2.477194
4        7        2.720603
5        8        2.577778


**Task 6 (drill).** Use `transform` to add a column `q_avg_alcohol` = the mean alcohol of each wine's quality group.

In [30]:
# Task 6 Code
wine["q_avg_alcohol"] = wine.groupby("quality")["alcohol"].transform("mean")
print(wine[["quality", "alcohol", "q_avg_alcohol"]].head())

   quality  alcohol  q_avg_alcohol
0        5      9.4       9.899706
1        5      9.8       9.899706
2        5      9.8       9.899706
3        6      9.8      10.629519
4        5      9.4       9.899706


**Task 7 (drill).** Build a `pivot_table` of mean `alcohol` with `quality` as the index (single column is fine).

In [31]:
# Task 7 Code
pivot_alc = wine.pivot_table(values="alcohol", index="quality", aggfunc="mean")
print(pivot_alc)

           alcohol
quality           
3         9.955000
4        10.265094
5         9.899706
6        10.629519
7        11.465913
8        12.094444


**Task 8 (challenge).** Using `quality_label`, compare high vs low wines with a multi-function `.agg()` on mean alcohol, mean volatile acidity, mean citric acid, and count. One sentence: what distinguishes high-quality wines?

In [32]:
# Task 8 Code
q_summary = wine.groupby("quality_label").agg(
    mean_alcohol=("alcohol", "mean"),
    mean_volatile_acidity=("volatile acidity", "mean"),
    mean_citric_acid=("citric acid", "mean"),
    count=("quality", "count")
)
print(q_summary)

               mean_alcohol  mean_volatile_acidity  mean_citric_acid  count
quality_label                                                              
high              11.518049               0.405530          0.376498    217
low               10.251037               0.547022          0.254407   1382


**Task 9 (challenge).** Build a reference table mapping quality 3–8 to a text `grade`, `merge` it onto `wine`, and show counts per grade.

In [ ]:

grade_map = pd.DataFrame({
    "quality": [3, 4, 5, 6, 7, 8],
    "grade": ["Poor", "Fair", "Average", "Good", "Very Good", "Excellent"]
})

wine = pd.merge(wine, grade_map, on="quality", how="left")
print(wine["grade"].value_counts())

grade
Average      681
Good         638
Very Good    199
Fair          53
Excellent     18
Poor          10
Name: count, dtype: int64


**Task 10 (challenge).** Create a binary column `high_alcohol` (1 if alcohol > median else 0). Then build a `pivot_table` of mean `quality` with `quality_label` rows and `high_alcohol` columns.

In [ ]:
# Your code here


**Task 11 (challenge).** Split the wines into two DataFrames (quality<=5 and quality>=6), add a `tier` column to each ('lower'/'upper'), then `concat` them back and confirm the row count matches the original.

In [ ]:
# Your code here


---
# Lecture 4 — Seaborn & the EDA Workflow

**EDA** = getting to know data before modelling. Explore in order:

1. **Univariate** — one variable (distribution, outliers)
2. **Bivariate** — two variables (relationships)
3. **Multivariate** — many variables (correlation, patterns)

In [ ]:
import seaborn as sns, matplotlib.pyplot as plt, pandas as pd, numpy as np
sns.set_theme(style="whitegrid")
tips = sns.load_dataset("tips")
tips.head()

## 4.0 First Look

In [ ]:
print("shape:", tips.shape)
print("missing:\n", tips.isnull().sum())
tips.describe()

## 4.1 Univariate

In [ ]:
sns.histplot(data=tips, x="total_bill", kde=True)
plt.title("total_bill distribution"); plt.show()

In [ ]:
sns.boxplot(data=tips, x="total_bill")
plt.title("total_bill boxplot (spot outliers)"); plt.show()

In [ ]:
sns.violinplot(data=tips, x="day", y="total_bill")
plt.title("violin = box + density"); plt.show()

In [ ]:
sns.countplot(data=tips, x="day")
plt.title("records per day"); plt.show()

In [ ]:
# Quick numeric skew check
print(tips[["total_bill", "tip"]].skew())

## 4.2 Bivariate

In [ ]:
sns.scatterplot(data=tips, x="total_bill", y="tip")
plt.title("tip vs total_bill"); plt.show()

In [ ]:
sns.boxplot(data=tips, x="day", y="total_bill")
plt.title("total_bill by day"); plt.show()

In [ ]:
sns.barplot(data=tips, x="day", y="tip")
plt.title("mean tip by day (with CI)"); plt.show()

In [ ]:
# Regression line for a numeric-numeric relationship
sns.lmplot(data=tips, x="total_bill", y="tip", height=4)
plt.title("lmplot: trend line"); plt.show()

## 4.3 Multivariate

In [ ]:
sns.scatterplot(data=tips, x="total_bill", y="tip", hue="sex", style="smoker")
plt.title("bill vs tip by sex & smoker"); plt.show()

### Correlation matrix + heatmap

In [ ]:
corr = tips.select_dtypes("number").corr()
print(corr)
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("correlation heatmap"); plt.show()

In [ ]:
# FacetGrid: same plot split across categories
g = sns.FacetGrid(tips, col="time", row="smoker", height=3)
g.map(sns.scatterplot, "total_bill", "tip")
plt.show()

In [ ]:
sns.pairplot(tips, hue="sex")
plt.show()

⚠️ **Common Mistake 10.** `.corr()` only works on numeric columns — pass `numeric_only=True` (or select numerics first) or newer pandas will error on string columns. Also remember correlation measures **linear** association only; a strong curved relationship can show a near-zero correlation.

⚠️ **Common Mistake 11.** Forgetting `plt.show()` (or drawing two plots in one cell without a new figure) can stack plots on top of each other. Start a fresh figure with `plt.figure()` when needed.

---
## 🧪 Lecture 4 Tasks — Seaborn & EDA (Wine Quality)

Same `winequality-red.csv` (semicolon-separated). Run a mini EDA: uni → bi → multivariate.

**Drills:** 1–7  •  **Challenges:** 8–11


In [ ]:
import pandas as pd, numpy as np, seaborn as sns, matplotlib.pyplot as plt
sns.set_theme(style="whitegrid")
wine = pd.read_csv("winequality-red.csv", sep=";")
print("shape:", wine.shape); wine.head()

**Task 1 (drill · uni).** Histogram + KDE of `alcohol`. Symmetric or skewed?

In [ ]:
# Your code here


**Task 2 (drill · uni).** `countplot` of `quality`. Most common score?

In [ ]:
# Your code here


**Task 3 (drill · uni).** Boxplot of `residual sugar` to inspect outliers. Print how many values exceed Q3 + 1.5*IQR.

In [ ]:
# Your code here


**Task 4 (drill · bi).** Boxplot of `alcohol` by `quality`. Does higher quality mean higher alcohol?

In [ ]:
# Your code here


**Task 5 (drill · bi).** Scatterplot of `volatile acidity` (x) vs `quality` (y). Note the trend.

In [ ]:
# Your code here


**Task 6 (drill · bi).** `barplot` of mean `alcohol` per `quality` score.

In [ ]:
# Your code here


**Task 7 (drill · multi).** Print the correlation of every feature with `quality`, sorted by absolute value.

In [ ]:
# Your code here


**Task 8 (challenge · multi).** Full correlation heatmap (annot, diverging cmap). Which two features correlate most strongly with `quality`?

In [ ]:
# Your code here


**Task 9 (challenge · multi).** Add `quality_label` (high if quality>=7 else low). Scatter `alcohol` vs `volatile acidity` coloured by label, then print group means of alcohol, volatile acidity, sulphates. Summarise in 1–2 sentences.

In [ ]:
# Your code here


**Task 10 (challenge · multi).** Make a `pairplot` of `alcohol`, `volatile acidity`, `sulphates`, `quality` coloured by `quality_label`. Which pair separates the classes best (visually)?

In [ ]:
# Your code here


**Task 11 (challenge · multi).** Use `pd.cut` to bin `alcohol` into 3 levels (low/med/high). Then make a grouped `barplot` of mean `quality` for each alcohol level split by `quality_label` hue. Comment on the pattern.

In [ ]:
# Your code here
